# VITS Inference Notebook for EmoNarrify

This notebook keeps the official VITS inference flow, but skips installation steps and uses the local paths in this workspace.

It includes:
- LJS single-speaker pretrained inference
- VCTK multi-speaker pretrained inference

All generated audio is saved under `outputs/notebook_inference/`.

In [2]:
import os
import sys
from pathlib import Path

import IPython.display as ipd
import torch
from scipy.io.wavfile import write

PROJECT_ROOT = Path('/home/ubuntu/emonarrify')
VITS_REPO = PROJECT_ROOT / 'vits'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook_inference'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(VITS_REPO)
sys.path.insert(0, str(VITS_REPO))

import commons
import utils
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence

def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    return torch.LongTensor(text_norm)

def synthesize_and_save(net_g, hps, text, out_path, sid=None, noise_scale=0.667, noise_scale_w=0.8, length_scale=1.0):
    stn_tst = get_text(text, hps)
    x_tst = stn_tst.unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)])

    with torch.no_grad():
        if sid is None:
            audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale)[0][0, 0].data.float().cpu().numpy()
        else:
            audio = net_g.infer(x_tst, x_tst_lengths, sid=sid, noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale)[0][0, 0].data.float().cpu().numpy()

    write(str(out_path), hps.data.sampling_rate, audio.astype('float32'))
    print(f'Saved: {out_path}')
    return audio

## Single Speaker: LJS pretrained

This section loads `weights/vits_pretrained/vits/pretrained_ljs.pth` and synthesizes a sample from the same sentence you can compare against the multi-speaker model.

In [4]:
TEXT = "We are going to have presentation tomorrow"

hps = utils.get_hparams_from_file(str(VITS_REPO / 'configs' / 'ljs_base.json'))
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model
)
_ = net_g.eval()
_ = utils.load_checkpoint(str(PROJECT_ROOT / 'weights' / 'vits_pretrained' / 'vits' / 'pretrained_ljs.pth'), net_g, None)

audio_ljs = synthesize_and_save(
    net_g=net_g,
    hps=hps,
    text=TEXT,
    out_path=OUTPUT_DIR / 'pretrained_ljs_longtext.wav',
)
ipd.display(ipd.Audio(audio_ljs, rate=hps.data.sampling_rate))

INFO:root:dec.cond.weight is not in the checkpoint
INFO:root:dec.cond.bias is not in the checkpoint
INFO:root:enc_q.enc.cond_layer.bias is not in the checkpoint
INFO:root:enc_q.enc.cond_layer.weight_g is not in the checkpoint
INFO:root:enc_q.enc.cond_layer.weight_v is not in the checkpoint
INFO:root:flow.flows.0.enc.cond_layer.bias is not in the checkpoint
INFO:root:flow.flows.0.enc.cond_layer.weight_g is not in the checkpoint
INFO:root:flow.flows.0.enc.cond_layer.weight_v is not in the checkpoint
INFO:root:flow.flows.2.enc.cond_layer.bias is not in the checkpoint
INFO:root:flow.flows.2.enc.cond_layer.weight_g is not in the checkpoint
INFO:root:flow.flows.2.enc.cond_layer.weight_v is not in the checkpoint
INFO:root:flow.flows.4.enc.cond_layer.bias is not in the checkpoint
INFO:root:flow.flows.4.enc.cond_layer.weight_g is not in the checkpoint
INFO:root:flow.flows.4.enc.cond_layer.weight_v is not in the checkpoint
INFO:root:flow.flows.6.enc.cond_layer.bias is not in the checkpoint
INFO:

## Multi Speaker: VCTK pretrained

This section loads `weights/vits_pretrained/vits/pretrained_vctk.pth` and uses a speaker id so you can compare multi-speaker inference quality.

In [3]:
hps_ms = utils.get_hparams_from_file(str(VITS_REPO / 'configs' / 'vctk_base.json'))
net_g_ms = SynthesizerTrn(
    len(symbols),
    hps_ms.data.filter_length // 2 + 1,
    hps_ms.train.segment_size // hps_ms.data.hop_length,
    n_speakers=hps_ms.data.n_speakers,
    **hps_ms.model
)
_ = net_g_ms.eval()
_ = utils.load_checkpoint(str(PROJECT_ROOT / 'weights' / 'vits_pretrained' / 'vits' / 'pretrained_vctk.pth'), net_g_ms, None)

sid = torch.LongTensor([4])  # speaker identity
audio_vctk = synthesize_and_save(
    net_g=net_g_ms,
    hps=hps_ms,
    text=TEXT,
    sid=sid,
    out_path=OUTPUT_DIR / 'pretrained_vctk_sid4_longtext.wav',
)
ipd.display(ipd.Audio(audio_vctk, rate=hps_ms.data.sampling_rate))

INFO:root:Loaded checkpoint '/home/ubuntu/emonarrify/weights/vits_pretrained/vits/pretrained_vctk.pth' (iteration 0)
Saved: /home/ubuntu/emonarrify/outputs/notebook_inference/pretrained_vctk_sid4_longtext.wav


## Saved outputs

Check the `outputs/notebook_inference/` folder for the generated WAV files.

In [4]:
for wav_path in sorted(OUTPUT_DIR.glob('*.wav')):
    print(wav_path)

/home/ubuntu/emonarrify/outputs/notebook_inference/pretrained_ljs_longtext.wav
/home/ubuntu/emonarrify/outputs/notebook_inference/pretrained_vctk_sid4_longtext.wav
